# Part 1


In [2]:
import pandas as pd

## Task 1


In [3]:
bike_rentals_raw = pd.read_csv("../bike_rentals.csv")
bike_rentals = bike_rentals_raw.copy()

Shape should be (403, 15). 'text' is 'str'.

In [5]:
print(bike_rentals.shape)
print(bike_rentals.dtypes)

(403, 15)
rental_id            str
station              str
bike_type            str
helmets_text         str
is_member            str
insured              str
paid_online          str
status               str
coupon_code      float64
damage_notes     float64
late_fee_note    float64
start_date           str
end_date             str
price                str
rating           float64
dtype: object


Columns where every value is missing. Should be coupon_code, damage_notes and late_fee_note.

In [7]:
missing_columns = bike_rentals.columns[bike_rentals.isna().all()]

print(list(missing_columns))

['coupon_code', 'damage_notes', 'late_fee_note']


Result after removal of empty columns. Should give (403, 12) and 0 empty columns.

In [12]:
#Do the removal, and then print it out.
bike_rentals = bike_rentals.dropna(axis=1, how='all')

print(bike_rentals.shape)

(403, 12)


In [16]:
missing_columns_after = bike_rentals.columns[bike_rentals.isna().all()]
print(list(missing_columns_after))
print(bike_rentals.isna().all().sum())

[]
0


## Task 2

first rows of price. currency and a comma
replace "SEK ", ""
we must convert it to a string, and then we can replace the "," with ".", and reconvert it to a float.

then, we can write the first 3 with .head(3)

float, type check, desvcribe() on price. 

then we will choose the 3 most expensive rentals.

column kallas för "price".

In [17]:
#Looking at the first rows of 'price'.
print(bike_rentals['price'].head(3))

0      SEK 855.50
1    SEK 1,311.50
2      SEK 350.00
Name: price, dtype: str


Removal of distracting parts for further processing in the 'price' column.

In [22]:
price = bike_rentals['price'].str.replace("SEK ", "")
print(price.head(3).tolist())

#Thousands marker removal.
price = price.str.replace(",", "")
print(price.head(3).tolist())

['855.50', '1,311.50', '350.00']
['855.50', '1311.50', '350.00']


Should be float64 when described. Values of interest: 
count : 352.0
mean : 847.3
std : 633.4
min : 154.5
median : 668.2 / 668.3
max : 3401.0

In [ ]:
bike_rentals['price'] = price.astype(float)

print(bike_rentals['price'].dtype)
print(bike_rentals['price'].describe().round(1))

float64
count     352.0
mean      847.3
std       633.4
min       154.5
25%       406.0
50%       668.2
75%      1194.6
max      3401.0
Name: price, dtype: float64


Top 3 with specific values given. Should be:

187 - R-5282 | Cargo bike | Completed | 3401.0

49 - R-5209 | Cargo bike | Completed | 2979.5

270 - R-5378 | E-bike | Completed | 2941.0

In [ ]:
bike_rentals.sort_values('price', ascending=False).head(3)[['rental_id', 'bike_type', 'status', 'price']]

,rental_id,bike_type,status,price
187,R-5282,Cargo bike,Completed,3401.0
49,R-5209,Cargo bike,Completed,2979.5
270,R-5378,E-bike,Completed,2941.0


## Task 3

Missing amount and percentage should be 51 and 12.7%, respectively.

In [ ]:
print(bike_rentals['price'].isna().sum())
print((bike_rentals['price'].isna().mean() * 100).round(1))

51
12.7


[No price] x [Cancelled rental] : Crosstab
The 4 not cancelled without price are missing without any clear reason, likely unintentional.
The 47 cancelled without price are missing because they weren't paid for.

In [33]:
no_price = bike_rentals['price'].isna()
cancelled_rental = bike_rentals['status'] == 'Cancelled'

pd.crosstab(no_price, cancelled_rental)

status,False,True
price,,
False,352,0
True,4,47


Leaving them to be as they are for the moment; no greater incentive yet to remove any of these.

## Task 4

Columns containing 't' and 'f' (Boolean representation).
Should be 'is_member', 'insured' and 'paid_online'. 'status' will not be here, because it is not represented binarily. 
It has a given status message.

In [ ]:
tf_columns = [col for col in bike_rentals.columns if set(bike_rentals[col].dropna().unique()) <= {'t', 'f'}]

print(tf_columns)

['is_member', 'insured', 'paid_online']


Mapping.

In [40]:
bike_rentals['is_member'] = bike_rentals['is_member'].map({'t' : True, 'f' : False})
bike_rentals['insured'] = bike_rentals['insured'].map({'t' : True, 'f' : False})
bike_rentals['paid_online'] = bike_rentals['paid_online'].map({'t' : True, 'f' : False})

3 missing on insured, 0 on both is_member and paid_online.

In [43]:
print(bike_rentals[['is_member', 'insured', 'paid_online']].isna().sum())

is_member      0
insured        3
paid_online    0
dtype: int64


115 members, 288 non-members.

In [59]:
print(bike_rentals['is_member'].sum())
print((bike_rentals['is_member'] == False).sum())

115
288


Median comparison of price : Members vs. Non-members (+ N.o. rentals).
[248 count, 756 median] vs. [104 count, 558 median]
Members generally pay less.

In [60]:
bike_rentals.groupby('is_member')['price'].agg(['count', 'median']).round(0)

,count,median
is_member,,
False,248,756.0
True,104,558.0
